# Lecture 2: The Ingestion Bottleneck and Streaming

## 1. The Shuffle Buffer Mechanism
A global Spark `reduceByKey` across a remote 10TB S3 bucket is computationally unfeasible due to memory constraints.

Let dataset size be $D$ and buffer size be $K$. True uniform random sampling requires $\mathcal{O}(D)$ memory.
However, a sliding-window buffer shuffling approach reduces this to $\mathcal{O}(K)$ memory, yielding pseudo-randomness sufficient for Stochastic Gradient Descent (SGD).

**Interactive Demonstration:** Run the cell below to simulate fetching data from a highly skewed stream (where data is sorted completely by class). Observe how increasing $K$ dilutes the correlation and generates a uniform random distribution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# A skewed streaming dataset where classes arrive sequentially (1000x 0s, 1000x 1s...)
stream_labels = np.repeat(np.arange(10), 1000)

def simulate_shuffle(buffer_size=10):
    np.random.seed(42)
    buffer = list(stream_labels[:buffer_size])
    stream_idx = buffer_size
    
    yielded = []
    
    # Simulate reading 1000 items from the pipeline
    for _ in range(1000):
        if not buffer:
            break
        pop_idx = np.random.randint(0, len(buffer))
        yielded.append(buffer.pop(pop_idx))
        if stream_idx < len(stream_labels):
            buffer.append(stream_labels[stream_idx])
            stream_idx += 1
            
    fig, ax = plt.subplots(figsize=(8, 4))
    counts = np.bincount(yielded, minlength=10)
    
    # Plotting
    ax.bar(np.arange(10), counts, color='#2ecc71', edgecolor='black')
    ax.axhline(100, color='r', linestyle='--', label='Perfect Uniform Target')
    ax.set_title(f'Sampled Class Distribution over 1000 items (Buffer Size K={buffer_size})')
    ax.set_ylim(0, 1000)
    ax.set_xlabel('Class Label')
    ax.set_ylabel('Yielded Count')
    ax.legend()
    plt.tight_layout()
    plt.show()

interact_ui = widgets.interactive(
    simulate_shuffle,
    buffer_size=widgets.IntSlider(min=1, max=5000, step=100, value=10, description='Buffer (K):')
)
display(interact_ui)

## 2. Optimizing the PyTorch DataLoader
The neural network is merely a consumer; the data engineer's job is to guarantee it is fed fast enough. We construct a `DataLoader` to optimize `batch_size`, `num_workers` (parallel downloading), and `pin_memory` (GPU page-locking).

In [ ]:
import webdataset as wds
import torch
import time
from torch.utils.data import DataLoader

url = "http://127.0.0.1:9000/cifar-streaming/cifar-train-{000000..000049}.tar"

# Build pipeline, actively shuffling the tar shards themselves and using a local shuffle buffer of 1000.
dataset = (
    wds.WebDataset(url, shardshuffle=True)
    .shuffle(1000)
    .decode("rgb")
    .to_tuple("png", "cls")
)

loader = DataLoader(
    dataset,
    batch_size=256,
    num_workers=4,       # Background fetchers
    pin_memory=True      # Allows fast transfer to CUDA Devices
)

print("DataLoader bound to stream successfully.")

## 3. Model Architecture Deep Dive: Small ResNet
Transitioning from MapReduce topologies to deep neural nets.
We define a residual mapping for the ResNet architecture:
$$y = \mathcal{F}(x, \{W_i\}) + x$$

The derivative allows gradient flow to bypass the non-linear blocks without vanishing:
$$\frac{\partial \mathcal{L}}{\partial x} = \frac{\partial \mathcal{L}}{\partial y} \left( \frac{\partial \mathcal{F}}{\partial x} + 1 \right)$$


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class ResNet9(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        
        # Residual Block
        self.res_conv1 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.res_conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(64 * 16 * 16, 10)

    def forward(self, x):
        # Base Path
        out = F.relu(self.bn1(self.conv1(x)))
        
        # Residual Mapping: F(x) + x
        res = self.res_conv1(out)
        res = F.relu(res)
        res = self.res_conv2(res)
        out = F.relu(out + res) # The Addition
        
        out = self.pool(out)
        out = out.reshape(out.size(0), -1)
        return self.fc(out)

# Instantiate the model
model = ResNet9()
print("ResNet9 architecture constructed.")

## 4. The Consumer: Active PyTorch Lightning Training
We map our model into a LightningModule computing Categorical Cross-Entropy:
$$\mathcal{L} = -\sum_{c=1}^{C} y_c \log(\hat{y}_c)$$

We will now **actively train** the model over the streamed DataLoader directly from MinIO!

In [ ]:
import pytorch_lightning as pl
import torch.optim as optim

class TrafficLight(pl.LightningModule):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        images, labels = batch
        if images.dtype == torch.uint8:
            images = images.float() / 255.0
            
        if images.shape[-1] == 3: # (N, H, W, C) -> (N, C, H, W)
            images = images.permute(0, 3, 1, 2).contiguous()
            
        outputs = self(images)
        loss = F.cross_entropy(outputs, labels)
        
        preds = torch.argmax(outputs, dim=1)
        acc = (preds == labels).float().mean()
        
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', acc, prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        images, labels = batch
        if images.dtype == torch.uint8:
            images = images.float() / 255.0
        if images.shape[-1] == 3:
            images = images.permute(0, 3, 1, 2).contiguous()
            
        outputs = self(images)
        loss = F.cross_entropy(outputs, labels)
        preds = torch.argmax(outputs, dim=1)
        acc = (preds == labels).float().mean()
        self.log('test_loss', loss, prog_bar=True)
        self.log('test_acc', acc, prog_bar=True)

    def configure_optimizers(self):
        return optim.AdamW(self.model.parameters(), lr=1e-3)

# Limit batches to fit comfortably within a live lecture
trainer = pl.Trainer(max_epochs=1, limit_train_batches=100, limit_test_batches=20, accelerator='auto')
lightning_model = TrafficLight(model)

print("🚀 Starting Training Loop over Streaming Datasets...")
trainer.fit(lightning_model, loader)

print("🧪 Starting Evaluation Phase...")
trainer.test(lightning_model, loader)